# Sanity check of Job 1 results

## Correctness of the results on the `flight_10.parquet` dataset

Defines libraries and starts a spark session:

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, min, max, count
import os
import re

spark = SparkSession.builder.appName("Sanity Check Job 1").getOrCreate()

**Loads** the **Parquet dataset**:

In [4]:
f10_relative_path = "../dataset/processed/flight_10.parquet"
f10_absolute_path = os.path.abspath(f10_relative_path)
f10_file_path = f"file://{f10_absolute_path}"
df = spark.read.parquet(f10_file_path)

**Reads** the **Hadoop results** file `job1_v1/flight_10`:

In [5]:
hadoop_path = os.path.abspath("../results/job1_mapreduce/job1_v1/flight_10/part-r-00000")
hadoop_line = ""

**Defines the <key,value> pair** – (Delta Airlines from Atlanta):

In [6]:
route = "DL,ATL"
carrier, origin = route.split(",")

**Filters the Parquet dataset** based on a specific <key,value> pair:

In [7]:
route_df = df.filter((col("op_unique_carrier") == carrier) & (col("origin") == origin))

**Calculates** metrics **and extracts** results **from the Parquet dataset**:

In [8]:
metrics_df = route_df.filter(col("cancelled") == 0).select(
                                                    count("arr_delay").alias("Valid_Flights"),
                                                    min("arr_delay").alias("Min_Delay"),
                                                    max("arr_delay").alias("Max_Delay"),
                                                    avg("arr_delay").alias("Avg_Delay")
                                                )

spark_row = metrics_df.collect()[0]
spark_valid_flights = spark_row["Valid_Flights"]
spark_min = spark_row["Min_Delay"]
spark_max = spark_row["Max_Delay"]
spark_avg = spark_row["Avg_Delay"]

**Filters Hadoop results** based on a specific <key,value> pair:

In [9]:
with open(hadoop_path, 'r') as file:
    for line in file:
        if line.startswith(route):
            hadoop_line = line
            break

**Makes the comparison**:

In [10]:
if hadoop_line:
    match = re.search(r"Flights Operated: (\d+) \| Min Arr Delay: ([-\d.]+) min \| Max Arr Delay: ([-\d.]+) min \| Avg Arr Delay: ([-\d.]+) min", hadoop_line)
    
    if match:
        hadoop_total_flights = int(match.group(1))
        hadoop_min = float(match.group(2))
        hadoop_max = float(match.group(3))
        hadoop_avg = float(match.group(4))
        
        print(f"--- CORRECTNESS CHECK FOR <{route}> ---")
        
        print(f"Min Delay:  Spark={spark_min:.2f} | Hadoop={hadoop_min:.2f} -> " + ("OK" if round(spark_min, 2) == hadoop_min else "ERROR"))
        print(f"Max Delay:  Spark={spark_max:.2f} | Hadoop={hadoop_max:.2f} -> " + ("OK" if round(spark_max, 2) == hadoop_max else "ERROR"))
        
        # 0.01 tollerance for the avg
        diff_avg = abs(spark_avg - hadoop_avg)
        print(f"Avg Delay:  Spark={spark_avg:.2f} | Hadoop={hadoop_avg:.2f} -> " + ("OK" if diff_avg <= 0.01 else "ERROR"))

--- CORRECTNESS CHECK FOR <DL,ATL> ---
Min Delay:  Spark=-59.00 | Hadoop=-59.00 -> OK
Max Delay:  Spark=755.00 | Hadoop=755.00 -> OK
Avg Delay:  Spark=5.41 | Hadoop=5.41 -> OK


Check on a **less busy airport** - SkyWest Airlines from Fargo:

In [15]:
route = "OO,FAR"
carrier, origin = route.split(",")

route_df = df.filter((col("op_unique_carrier") == carrier) & (col("origin") == origin))

metrics_df = route_df.filter(col("cancelled") == 0).select(
                                                    count("arr_delay").alias("Valid_Flights"),
                                                    min("arr_delay").alias("Min_Delay"),
                                                    max("arr_delay").alias("Max_Delay"),
                                                    avg("arr_delay").alias("Avg_Delay")
                                                )

spark_row = metrics_df.collect()[0]
spark_valid_flights = spark_row["Valid_Flights"]
spark_min = spark_row["Min_Delay"]
spark_max = spark_row["Max_Delay"]
spark_avg = spark_row["Avg_Delay"]

with open(hadoop_path, 'r') as file:
    for line in file:
        if line.startswith(route):
            hadoop_line = line
            break

if hadoop_line:
    match = re.search(r"Flights Operated: (\d+) \| Min Arr Delay: ([-\d.]+) min \| Max Arr Delay: ([-\d.]+) min \| Avg Arr Delay: ([-\d.]+) min", hadoop_line)
    
    if match:
        hadoop_total_flights = int(match.group(1))
        hadoop_min = float(match.group(2))
        hadoop_max = float(match.group(3))
        hadoop_avg = float(match.group(4))
        
        print(f"--- CORRECTNESS CHECK FOR <{route}> ---")
        
        print(f"Min Delay:  Spark={spark_min:.2f} | Hadoop={hadoop_min:.2f} -> " + ("OK" if round(spark_min, 2) == hadoop_min else "ERROR"))
        print(f"Max Delay:  Spark={spark_max:.2f} | Hadoop={hadoop_max:.2f} -> " + ("OK" if round(spark_max, 2) == hadoop_max else "ERROR"))
        
        # 0.01 tollerance for the avg
        diff_avg = abs(spark_avg - hadoop_avg)
        print(f"Avg Delay:  Spark={spark_avg:.2f} | Hadoop={hadoop_avg:.2f} -> " + ("OK" if diff_avg <= 0.01 else "ERROR"))

--- CORRECTNESS CHECK FOR <OO,FAR> ---
Min Delay:  Spark=-45.00 | Hadoop=-45.00 -> OK
Max Delay:  Spark=1125.00 | Hadoop=1125.00 -> OK
Avg Delay:  Spark=20.66 | Hadoop=20.66 -> OK


## Consistency of the scalability of the results

**Defines** the **Hadoop results file paths**:

In [16]:
files = {
    100: "../results/job1_mapreduce/job1_v1/flight_100/part-r-00000",
    200: "../results/job1_mapreduce/job1_v1/flight_200/part-r-00000",
    300: "../results/job1_mapreduce/job1_v1/flight_300/part-r-00000"
}

**Calculates and extracts the metrics** through helper function:

In [17]:
metrics = {}

for percentage, relative_path in files.items():
    absolute_path = os.path.abspath(relative_path)
    
    # Inizializziamo a None in caso il file non esista
    metrics[percentage] = {"flights": None, "avg": None}
    
    try:
        with open(absolute_path, 'r') as file:
            for line in file:
                if line.startswith(route):
                    match = re.search(r"Flights Operated: (\d+).*Avg Arr Delay: ([-\d.]+) min", line)
                    if match:
                        metrics[percentage]["flights"] = int(match.group(1))
                        metrics[percentage]["avg"] = float(match.group(2))
                        break
    except FileNotFoundError:
        print(f" ERROR: File per il {percentage}% non trovato in {absolute_path}")

v100, m100 = metrics[100]["flights"], metrics[100]["avg"]
v200, m200 = metrics[200]["flights"], metrics[200]["avg"]
v300, m300 = metrics[300]["flights"], metrics[300]["avg"]

**Validation**:

In [18]:
if v100 is not None and v200 is not None and v300 is not None:
    
    check_200_count = (v200 == v100 * 2)
    check_300_count = (v300 == v100 * 3)

    print(f"--- CONSISTENCY OF THE SCALABILITY CHECK FOR <{route}> ---")
    
    print(f"flights (Base 100%): {v100}")
    print(f"flights (Base 200%): {v200} (Exp: {v100 * 2}) -> " + ("OK" if check_200_count else "ERROR"))
    print(f"flights (Base 300%): {v300} (Exp: {v100 * 3}) -> " + ("OK" if check_300_count else "ERROR"))
    
    print("-" * 30)
    
    # Verifica Medie (devono restare identiche)
    check_200_avg = (m100 == m200)
    check_300_avg = (m100 == m300)
    
    print(f"avg (Base 100%): {m100} min")
    print("  Exp invariance")
    print(f"avg (Base 200%): {m200} min -> " + ("OK" if check_200_avg else "ERROR"))
    print(f"avg (Base 300%): {m300} min -> " + ("OK" if check_300_avg else "ERROR"))

--- CONSISTENCY OF THE SCALABILITY CHECK FOR <OO,FAR> ---
flights (Base 100%): 2859
flights (Base 200%): 5718 (Exp: 5718) -> OK
flights (Base 300%): 8577 (Exp: 8577) -> OK
------------------------------
avg (Base 100%): 13.8 min
  Exp invariance
avg (Base 200%): 13.8 min -> OK
avg (Base 300%): 13.8 min -> OK


# Sanity check of Job 1 v2 results

## Correctness of the results on the `flight_10.parquet` dataset

**Reads** the **Hadoop results** file `/job1_v2/flight_10`:

In [19]:
hadoop_path = os.path.abspath("../results/job1_mapreduce/job1_v2/flight_10/part-r-00000")
hadoop_line = ""

In [20]:
route = "DL,ATL-JFK"
carrier, section = route.split(",")
origin, dest = section.split("-")

In [21]:
route_df = df.filter((col("op_unique_carrier") == carrier) & (col("origin") == origin) & (col("dest") == dest))

In [24]:
metrics_df = route_df.filter(col("cancelled") == 0).select(
                                                    count("arr_delay").alias("Valid_Flights"),
                                                    min("arr_delay").alias("Min_Delay"),
                                                    max("arr_delay").alias("Max_Delay"),
                                                    avg("arr_delay").alias("Avg_Delay")
                                                )

spark_row = metrics_df.collect()[0]
spark_valid_flights = spark_row["Valid_Flights"]
spark_min = spark_row["Min_Delay"]
spark_max = spark_row["Max_Delay"]
spark_avg = spark_row["Avg_Delay"]

In [25]:
with open(hadoop_path, 'r') as file:
    for line in file:
        if line.startswith(route):
            hadoop_line = line
            break

In [26]:
if hadoop_line:
    match = re.search(r"Flights Operated: (\d+) \| Min Arr Delay: ([-\d.]+) min \| Max Arr Delay: ([-\d.]+) min \| Avg Arr Delay: ([-\d.]+) min", hadoop_line)
    
    if match:
        hadoop_total_flights = int(match.group(1))
        hadoop_min = float(match.group(2))
        hadoop_max = float(match.group(3))
        hadoop_avg = float(match.group(4))
        
        print(f"--- CORRECTNESS CHECK FOR <{route}> ---")
        
        print(f"Min Delay:  Spark={spark_min:.2f} | Hadoop={hadoop_min:.2f} -> " + ("OK" if round(spark_min, 2) == hadoop_min else "ERROR"))
        print(f"Max Delay:  Spark={spark_max:.2f} | Hadoop={hadoop_max:.2f} -> " + ("OK" if round(spark_max, 2) == hadoop_max else "ERROR"))
        
        # 0.01 tollerance for the avg
        diff_avg = abs(spark_avg - hadoop_avg)
        print(f"Avg Delay:  Spark={spark_avg:.2f} | Hadoop={hadoop_avg:.2f} -> " + ("OK" if diff_avg <= 0.01 else "ERROR"))

--- CORRECTNESS CHECK FOR <DL,ATL-JFK> ---
Min Delay:  Spark=-40.00 | Hadoop=-40.00 -> OK
Max Delay:  Spark=428.00 | Hadoop=428.00 -> OK
Avg Delay:  Spark=10.56 | Hadoop=10.56 -> OK
